In [ ]:
# Mount Drive & navigate to project ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# cloned the project
PROJECT_DIR = '/content/'
!git clone https://github.com/VanAnh2240/aurafit_personal_color.git

Cloning into 'aurafit_personal_color'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 33 (delta 2), reused 33 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 49.85 KiB | 4.98 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
import os
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

Working directory: /content
Files: ['.config', 'drive', 'aurafit_personal_color', 'sample_data']


In [ ]:
# unzip dataset
!tar -xzvf "/content/drive/MyDrive/1701_FashionGen/aurafit_personal_color/LaPa.tar.gz" -C /content/

Streaming output truncated to the last 5000 lines.
LaPa/train/labels/2818584124_13.png
LaPa/train/labels/LFPW_image_test_0157_1.png
LaPa/train/labels/4595280121_4.png
LaPa/train/labels/HELEN_2761106136_1_3.png
LaPa/train/labels/HELEN_3005087184_1_0.png
LaPa/train/labels/5906318280_1.png
LaPa/train/labels/HELEN_126968967_1_0.png
LaPa/train/labels/LFPW_image_test_0157_2.png
LaPa/train/labels/AFW_5002723411_2_9.png
LaPa/train/labels/LFPW_image_train_0292_6.png
LaPa/train/labels/12016442526_0.png
LaPa/train/labels/LFPW_image_train_0789_7.png
LaPa/train/labels/LFPW_image_train_0588_0.png
LaPa/train/labels/HELEN_173808384_2_5.png
LaPa/train/labels/6531382301_15.png
LaPa/train/labels/HELEN_2707642369_1_4.png
LaPa/train/labels/12268264184_13.png
LaPa/train/labels/71244599_2.png
LaPa/train/labels/6259612800_2.png
LaPa/train/labels/AFW_4906266640_1_1.png
LaPa/train/labels/HELEN_2414075021_1_0.png
LaPa/train/labels/HELEN_2233737284_1_3.png
LaPa/train/labels/HELEN_1419222657_1_1.png
LaPa/train/lab

In [ ]:
!ls /content/aurafit_personal_color

app		    download_lapa.py	preprocess.py	  src
colab_runner.ipynb  evaluate.py		README.md	  train.py
config.py	    prepare_dataset.py	requirements.txt


In [ ]:
!pip install -q albumentations scikit-image scikit-learn flask flask-cors pyngrok
!pip install -q git+https://github.com/openai/CLIP.git
print('Dependencies installed ✓')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
Dependencies installed ✓


In [ ]:
!ls /content/
%cd /content/aurafit_personal_color
!pip install -r requirements.txt

aurafit_personal_color	drive  LaPa  sample_data
/content/aurafit_personal_color
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-e7vlprro
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-e7vlprro
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done


In [ ]:
import sys; sys.path.insert(0, '.')
from src.dataset import dataset_summary
dataset_summary()


LaPa Dataset Summary
  train   images= 18168  labels=✓           landmarks= 18168
  val     images=  2000  labels=✓           landmarks=  2000
  test    images=  2000  labels=✓           landmarks=  2000
  train → supervised training  (image + label)
  val   → visual inspection only (image only, NO label)
  test  → final mIoU evaluation (image + label)



In [ ]:
%cd /content/aurafit_personal_color

/content/aurafit_personal_color


In [ ]:
!python prepare_dataset.py --raw_dir data/raw --train_ratio 0.9 --seed 42


LaPa raw layout detected:
  test/ (labeled) :   2000 images
  val/  (no labels):   2000 images

Split plan (seed=42, train_ratio=0.9):
  → train/  :   1800 images  (with labels)
  → test/   :    200 images  (with labels, eval only)
  → val/    : kept as-is   (no labels, visual inspection only)

Method: symlink (fast)

  train/ exists but has 18168 images (expected 1800). Rebuilding...

  Backing up original test/ → _test_original/ ...

  Building train/ ...
  Rebuilding test/ (eval subset) ...
  val/ kept as-is (no labels).

Final structure in /content/aurafit_personal_color/data/raw:
split       images    labels   landmarks
────────────────────────────────────────────
train         1800      1800        1800
val           2000      2000        2000
test           200       200         200

✅ Dataset ready.  Run:  python train.py --model clipunet --kfold



In [ ]:
!python train.py --model deeplab --epochs 50 --device auto


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Model   : deeplab
  Device  : cuda
  Epochs  : 50
  Mode    : Standard
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [dataset] train: 1800 labeled pairs loaded

  Train=1620  Monitor=180  (10 % labeled hold-out — LaPa val has no labels)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/aurafit_personal_color/train.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler   = torch.cuda.amp.GradScaler()

In [ ]:
!python train.py --model clipunet --epochs 50 --device auto


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Model   : clipunet
  Device  : cuda
  Epochs  : 50
  Mode    : Standard
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [dataset] train: 1800 labeled pairs loaded

  Train=1620  Monitor=180  (10 % labeled hold-out — LaPa val has no labels)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/aurafit_personal_color/train.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler   = torch.cuda.amp.GradScaler(

In [ ]:
!python train.py --model clipunet --epochs 100 --resume checkpoints/system_2_clipunet.pth

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [ ]:
!python evaluate.py --model clipunet --mode seg